In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

from scipy.stats import mannwhitneyu

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.passqc_nuclei.h5ad')

In [ ]:
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
genes_path = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations/ensembl_to_symbol.tsv'
genes = pd.read_csv(genes_path, sep='\t')
gene_dict = dict(zip(genes['gene'], genes['symbol']))

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', header=0, index_col=0)
ips_clusters.shape

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
adata.obs['ips_subtype'] = ips_clusters.reindex(adata.obs.index)['ips_subtype']
ips_adata = adata[~adata.obs['ips_subtype'].isna()]
ips_adata.shape

### grab canonical pluripotency markers

In [ ]:
genes = ['EPCAM', 'NANOG', 'DNMT3B', 'LIN28A']
genes = pd.Index(genes)
mask = genes.isin(adata.var.index) 
genes = genes[mask]
genes.shape

In [ ]:
indices = ips_adata.var.index.get_indexer(genes)
for g,idx in zip(genes, indices):
    ips_adata.obs[g] = ips_adata.X[:,idx].toarray()
    
tmp_meta = ips_adata.obs.copy()
tmp_meta.shape

In [ ]:
tmp_meta.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i,ax in enumerate(axes.flatten()):
    g = genes[i]
    sns.boxplot(tmp_meta, x='ips_subtype', y=g, hue='donor', ax=ax)
    sns.move_legend(ax, 'upper right', bbox_to_anchor=(1.3, 1.3))
    ax.set_title(g, fontsize=16)
    ax.set_ylabel('Log expr')
    
plt.subplots_adjust(hspace=0.5, wspace=0.5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8,8))

for i,ax in enumerate(axes.flatten()):
    g = genes[i]
    sns.boxplot(tmp_meta, x='donor', y=g, hue='ips_subtype', ax=ax, palette='Set3')
    sns.move_legend(ax, 'upper right', bbox_to_anchor=(1.3, 1.3))
    ax.set_title(g, fontsize=16)
    ax.set_ylabel('Log expr')
    
plt.subplots_adjust(hspace=0.5, wspace=0.5)

### mki67

In [ ]:
genes = ['MKI67']
genes = pd.Index(genes)
mask = genes.isin(adata.var.index) 
genes = genes[mask]

indices = ips_adata.var.index.get_indexer(genes)
for g,idx in zip(genes, indices):
    ips_adata.obs[g] = ips_adata.X[:,idx].toarray()
    
tmp_meta = ips_adata.obs.copy()
fig, ax = plt.subplots(1, figsize=(8,8))

i = 0
g = genes[i]
sns.boxplot(tmp_meta, x='ips_subtype', y=g, hue='donor', ax=ax)
sns.move_legend(ax, 'upper right', bbox_to_anchor=(1.3, 1.3))
ax.set_title(g, fontsize=16)
ax.set_ylabel('Log expr')

plt.subplots_adjust(hspace=0.5, wspace=0.5)

In [ ]:
genes = ['MKI67']
genes = pd.Index(genes)
mask = genes.isin(adata.var.index) 
genes = genes[mask]

indices = ips_adata.var.index.get_indexer(genes)
for g,idx in zip(genes, indices):
    ips_adata.obs[g] = ips_adata.X[:,idx].toarray()
    
tmp_meta = ips_adata.obs.copy()
fig, ax = plt.subplots(1, figsize=(8,8))

i = 0
g = genes[i]
sns.boxplot(tmp_meta, x='donor', y=g, hue='ips_subtype', ax=ax)
sns.move_legend(ax, 'upper right', bbox_to_anchor=(1.3, 1.3))
ax.set_title(g, fontsize=16)
ax.set_ylabel('Log expr')

plt.subplots_adjust(hspace=0.5, wspace=0.5)

### write

In [ ]:
genes = ['EPCAM', 'NANOG', 'DNMT3B', 'LIN28A']
to_plot = tmp_meta.melt(id_vars=['ips_subtype'], value_vars=genes)
to_plot.head()

In [ ]:
tmp_meta

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
to_plot.to_csv(f'{outdir}/ips_subtypes_pluri_markers.csv', sep='\t')

In [ ]:
!date